# Completion Notebook – Transaction-Cost-Aware Deep Hedging Model

This notebook completes the modelling part that is still missing.

The existing modelling notebook already provides:
- a baseline strategy trained with the quadratic loss,
- an asymmetric-loss strategy.

The remaining modelling block is the transaction-cost-aware strategy. This notebook trains an augmented LSTM where transaction costs are directly included in the training objective.

The main output is:
- `deltas_transaction_cost.npy`

This file can then be used in the financial evaluation notebook exactly like the other fixed hedging outputs.


## Expected files in the project root

This notebook assumes that the following files are available in the current project root:
- `dataset.pkl`
- `config.json`
- optionally `deltas_mse.npy`
- optionally `deltas_asymmetric.npy`

The optional files are only used for comparison. The transaction-cost model is trained from the dataset.


In [ ]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## File paths

In [ ]:
PROJECT_ROOT = Path(".").resolve()
OUTPUT_DIR = PROJECT_ROOT
RESULTS_DIR = PROJECT_ROOT / "arthur_completion_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

dataset_file = PROJECT_ROOT / "dataset.pkl"
config_file = PROJECT_ROOT / "config.json"

if not dataset_file.exists():
    raise FileNotFoundError(f"Missing required file: {dataset_file}")

if not config_file.exists():
    raise FileNotFoundError(f"Missing required file: {config_file}")

baseline_file = PROJECT_ROOT / "deltas_mse.npy"
asymmetric_file = PROJECT_ROOT / "deltas_asymmetric.npy"

print("Dataset:", dataset_file)
print("Config:", config_file)
print("Baseline deltas available:", baseline_file.exists())
print("Asymmetric deltas available:", asymmetric_file.exists())


## Load dataset and configuration

In [ ]:
with open(dataset_file, "rb") as f:
    dataset = pickle.load(f)

with open(config_file, "r", encoding="utf-8") as f:
    config = json.load(f)

train_state = np.array(dataset["train"]["state_paths"], dtype=np.float32)
train_tradable = np.array(dataset["train"]["tradable_paths"], dtype=np.float32)
train_payoff = np.array(dataset["train"]["payoff"], dtype=np.float32)

val_state = np.array(dataset["validation"]["state_paths"], dtype=np.float32)
val_tradable = np.array(dataset["validation"]["tradable_paths"], dtype=np.float32)
val_payoff = np.array(dataset["validation"]["payoff"], dtype=np.float32)

test_state = np.array(dataset["test"]["state_paths"], dtype=np.float32)
test_tradable = np.array(dataset["test"]["tradable_paths"], dtype=np.float32)
test_payoff = np.array(dataset["test"]["payoff"], dtype=np.float32)

time_days = np.array(dataset["time_days"], dtype=np.float32)

liquidity = np.array(config["liquidity"], dtype=np.float32)
transaction_costs = np.array(config["transaction_costs"], dtype=np.float32)

N = train_tradable.shape[1] - 1
d = train_tradable.shape[2]
state_dim = train_state.shape[2]

print("train state:", train_state.shape)
print("train tradable:", train_tradable.shape)
print("train payoff:", train_payoff.shape)
print("validation state:", val_state.shape)
print("test state:", test_state.shape)
print("N:", N, "d:", d, "state_dim:", state_dim)
print("liquidity:", liquidity)
print("transaction costs:", transaction_costs)


## Normalization

The state variables are normalized using statistics estimated on the training set. The same normalization is then applied to validation and test data.

This keeps the evaluation strictly out-of-sample and avoids using test-set information during training.


In [ ]:
state_mean = train_state.mean(axis=0, keepdims=True)
state_std = train_state.std(axis=0, keepdims=True)
state_std = np.maximum(state_std, 1e-8)

train_state_norm = (train_state - state_mean) / state_std
val_state_norm = (val_state - state_mean) / state_std
test_state_norm = (test_state - state_mean) / state_std

def to_tensor(x):
    return torch.tensor(x, dtype=torch.float32, device=DEVICE)

train_state_t = to_tensor(train_state_norm)
train_tradable_t = to_tensor(train_tradable)
train_payoff_t = to_tensor(train_payoff)

val_state_t = to_tensor(val_state_norm)
val_tradable_t = to_tensor(val_tradable)
val_payoff_t = to_tensor(val_payoff)

test_state_t = to_tensor(test_state_norm)
test_tradable_t = to_tensor(test_tradable)
test_payoff_t = to_tensor(test_payoff)

liq_t = to_tensor(liquidity)
cost_t = to_tensor(transaction_costs)


## Augmented LSTM architecture

The model receives the normalized state and the current hedge position. It outputs a bounded hedge increment. The cumulative position is then used to compute the self-financing gains.

The `tanh` activation enforces the liquidity bound on hedge increments.


In [ ]:
class TransactionCostAwareLSTM(nn.Module):
    def __init__(self, state_dim, d, lstm_units=50, ff_layers=3, ff_width=10, liquidity=None):
        super().__init__()
        self.state_dim = state_dim
        self.d = d
        self.lstm_units = lstm_units

        if liquidity is None:
            liquidity = torch.ones(d)
        self.register_buffer("liquidity", torch.tensor(liquidity, dtype=torch.float32))

        self.lstm = nn.LSTMCell(input_size=state_dim + d, hidden_size=lstm_units)

        layers = []
        input_dim = lstm_units
        for _ in range(ff_layers):
            layers.append(nn.Linear(input_dim, ff_width))
            layers.append(nn.ReLU())
            input_dim = ff_width
        layers.append(nn.Linear(input_dim, d))
        self.feedforward = nn.Sequential(*layers)

    def forward(self, state_norm, tradable_paths):
        batch_size, n_steps_plus_one, _ = state_norm.shape
        N = n_steps_plus_one - 1

        h = torch.zeros(batch_size, self.lstm_units, device=state_norm.device)
        c = torch.zeros(batch_size, self.lstm_units, device=state_norm.device)
        delta_prev = torch.zeros(batch_size, self.d, device=state_norm.device)

        deltas = []

        for t in range(N):
            x_t = torch.cat([state_norm[:, t, :], delta_prev], dim=1)
            h, c = self.lstm(x_t, (h, c))

            raw_increment = self.feedforward(h)
            delta_increment = self.liquidity * torch.tanh(raw_increment)
            delta_t = delta_prev + delta_increment

            deltas.append(delta_t)
            delta_prev = delta_t

        deltas = torch.stack(deltas, dim=1)

        price_increments = tradable_paths[:, 1:, :] - tradable_paths[:, :-1, :]
        gains = torch.sum(deltas * price_increments, dim=(1, 2))

        return gains, deltas


## Transaction-cost-aware loss

The model is trained on the net terminal hedging error:
\[
Y_T^{net} = X_T^\Delta - g(S_T) - C_T(\Delta).
\]

The objective minimizes:
\[
\mathbb{E}\left[(Y_T^{net})^2ight].
\]

This directly internalizes trading costs during training.


In [ ]:
def torch_transaction_costs(deltas, transaction_costs):
    first_trade = torch.abs(deltas[:, 0, :])
    later_trades = torch.abs(deltas[:, 1:, :] - deltas[:, :-1, :])
    all_trades = torch.cat([first_trade.unsqueeze(1), later_trades], dim=1)
    return torch.sum(all_trades * transaction_costs.view(1, 1, -1), dim=(1, 2))


def transaction_cost_aware_loss(gains, deltas, payoff, transaction_costs):
    costs = torch_transaction_costs(deltas, transaction_costs)
    net_error = gains - payoff - costs
    return torch.mean(net_error ** 2), net_error, costs


## Training loop

The training loop uses Adam and tracks both training and validation loss. The best model is selected according to validation loss.


In [ ]:
def make_dataloader(state_norm, tradable, payoff, batch_size):
    ds = torch.utils.data.TensorDataset(state_norm, tradable, payoff)
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)


def train_transaction_cost_model(
    model,
    train_state,
    train_tradable,
    train_payoff,
    val_state,
    val_tradable,
    val_payoff,
    transaction_costs,
    n_iter=10000,
    batch_size=50,
    lr=1e-3,
    checkpoint_every=500,
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = make_dataloader(train_state, train_tradable, train_payoff, batch_size)

    train_losses = []
    val_losses = []

    best_state = None
    best_val_loss = np.inf

    iterator = iter(loader)

    for iteration in range(1, n_iter + 1):
        try:
            batch_state, batch_tradable, batch_payoff = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch_state, batch_tradable, batch_payoff = next(iterator)

        model.train()
        optimizer.zero_grad()

        gains, deltas = model(batch_state, batch_tradable)
        loss, _, _ = transaction_cost_aware_loss(gains, deltas, batch_payoff, transaction_costs)

        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.detach().cpu()))

        if iteration % checkpoint_every == 0 or iteration == 1:
            model.eval()
            with torch.no_grad():
                val_gains, val_deltas = model(val_state, val_tradable)
                val_loss, _, _ = transaction_cost_aware_loss(
                    val_gains, val_deltas, val_payoff, transaction_costs
                )

            val_loss_float = float(val_loss.detach().cpu())
            val_losses.append((iteration, val_loss_float))

            if val_loss_float < best_val_loss:
                best_val_loss = val_loss_float
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }

            print(f"Iteration {iteration:6d} | train loss {train_losses[-1]:.6f} | val loss {val_loss_float:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, train_losses, val_losses, best_val_loss


## Train transaction-cost-aware model

The default number of iterations is set to 10,000 to remain consistent with the existing modelling notebook. If time is limited, reduce `N_ITER` to 3,000 or 5,000 for a faster run.


In [ ]:
N_ITER = 10000
BATCH_SIZE = 50
LR = 1e-3
LSTM_UNITS = 50
FF_LAYERS = 3
FF_WIDTH = 10

model_tc = TransactionCostAwareLSTM(
    state_dim=state_dim,
    d=d,
    lstm_units=LSTM_UNITS,
    ff_layers=FF_LAYERS,
    ff_width=FF_WIDTH,
    liquidity=liquidity
).to(DEVICE)

model_tc, train_losses_tc, val_losses_tc, best_val_loss_tc = train_transaction_cost_model(
    model=model_tc,
    train_state=train_state_t,
    train_tradable=train_tradable_t,
    train_payoff=train_payoff_t,
    val_state=val_state_t,
    val_tradable=val_tradable_t,
    val_payoff=val_payoff_t,
    transaction_costs=cost_t,
    n_iter=N_ITER,
    batch_size=BATCH_SIZE,
    lr=LR,
    checkpoint_every=500,
)

print("Best validation loss:", best_val_loss_tc)


## Training curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses_tc, label="Train loss", alpha=0.7)

if len(val_losses_tc) > 0:
    val_iters = [x[0] for x in val_losses_tc]
    val_values = [x[1] for x in val_losses_tc]
    plt.plot(val_iters, val_values, marker="o", label="Validation loss")

plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Transaction-cost-aware loss")
plt.title("Training curve: transaction-cost-aware model")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "transaction_cost_model_training_curve.png", dpi=150)
plt.show()


## Test evaluation and saved outputs

The main output is `deltas_transaction_cost.npy`, saved in the project root so that it can be loaded directly by the financial evaluation notebook.


In [ ]:
model_tc.eval()

with torch.no_grad():
    test_gains_tc, test_deltas_tc = model_tc(test_state_t, test_tradable_t)
    test_loss_tc, test_net_error_tc, test_costs_tc = transaction_cost_aware_loss(
        test_gains_tc, test_deltas_tc, test_payoff_t, cost_t
    )

deltas_transaction_cost = test_deltas_tc.detach().cpu().numpy()
net_errors_transaction_cost = test_net_error_tc.detach().cpu().numpy()
costs_transaction_cost = test_costs_tc.detach().cpu().numpy()
gross_errors_transaction_cost = (test_gains_tc - test_payoff_t).detach().cpu().numpy()

np.save(OUTPUT_DIR / "deltas_transaction_cost.npy", deltas_transaction_cost)
np.save(OUTPUT_DIR / "net_errors_transaction_cost.npy", net_errors_transaction_cost)
np.save(OUTPUT_DIR / "gross_errors_transaction_cost.npy", gross_errors_transaction_cost)
np.save(OUTPUT_DIR / "transaction_costs_model.npy", costs_transaction_cost)

torch.save(model_tc.state_dict(), OUTPUT_DIR / "model_transaction_cost.pt")

print("Saved:", OUTPUT_DIR / "deltas_transaction_cost.npy")
print("Test transaction-cost-aware loss:", float(test_loss_tc.detach().cpu()))
print("Deltas shape:", deltas_transaction_cost.shape)


## Metrics for the transaction-cost-aware model

In [ ]:
def empirical_cvar_left(x, alpha=0.05):
    threshold = np.quantile(x, alpha)
    tail = x[x <= threshold]
    return float(np.mean(tail))


def summarize_vector(x):
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "min": float(np.min(x)),
        "q05": float(np.quantile(x, 0.05)),
        "median": float(np.median(x)),
        "q95": float(np.quantile(x, 0.95)),
        "max": float(np.max(x)),
        "cvar05": empirical_cvar_left(x, 0.05)
    }


metrics_tc = {
    "model": "transaction_cost_aware",
    "mse_gross_error": float(np.mean(gross_errors_transaction_cost ** 2)),
    "mse_net_error": float(np.mean(net_errors_transaction_cost ** 2)),
    "mean_gross_error": float(np.mean(gross_errors_transaction_cost)),
    "mean_net_error": float(np.mean(net_errors_transaction_cost)),
    "std_net_error": float(np.std(net_errors_transaction_cost)),
    "net_q05": float(np.quantile(net_errors_transaction_cost, 0.05)),
    "net_cvar05": empirical_cvar_left(net_errors_transaction_cost, 0.05),
    "mean_transaction_cost": float(np.mean(costs_transaction_cost)),
}

with open(RESULTS_DIR / "transaction_cost_model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_tc, f, indent=2)

metrics_tc


## Optional comparison with existing baseline outputs

If `deltas_mse.npy` and `deltas_asymmetric.npy` are available, this block compares all strategies under the same net-PnL evaluation framework.


In [ ]:
def numpy_transaction_costs(deltas, transaction_costs):
    first_trade = np.abs(deltas[:, 0, :])
    later_trades = np.abs(deltas[:, 1:, :] - deltas[:, :-1, :])
    all_trades = np.concatenate([first_trade[:, None, :], later_trades], axis=1)
    return np.sum(all_trades * transaction_costs[None, None, :], axis=(1, 2))


def numpy_gross_error(tradable_paths, deltas, payoff):
    increments = tradable_paths[:, 1:, :] - tradable_paths[:, :-1, :]
    gains = np.sum(deltas * increments, axis=(1, 2))
    return gains - payoff


comparison_rows = []

available_models = {
    "transaction_cost_aware": deltas_transaction_cost
}

if baseline_file.exists():
    available_models["baseline_mse"] = np.load(baseline_file)

if asymmetric_file.exists():
    available_models["asymmetric_loss"] = np.load(asymmetric_file)

for name, deltas in available_models.items():
    gross_error = numpy_gross_error(test_tradable, deltas, test_payoff)
    costs = numpy_transaction_costs(deltas, transaction_costs)
    net_error = gross_error - costs
    turnover = np.mean(np.sum(np.abs(np.concatenate(
        [deltas[:, :1, :], deltas[:, 1:, :] - deltas[:, :-1, :]], axis=1
    )), axis=(1, 2)))

    comparison_rows.append({
        "model": name,
        "mse_gross_error": float(np.mean(gross_error ** 2)),
        "mse_net_error": float(np.mean(net_error ** 2)),
        "mean_net_error": float(np.mean(net_error)),
        "std_net_error": float(np.std(net_error)),
        "net_q05": float(np.quantile(net_error, 0.05)),
        "net_cvar05": empirical_cvar_left(net_error, 0.05),
        "mean_transaction_cost": float(np.mean(costs)),
        "mean_turnover": float(turnover),
    })

comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(RESULTS_DIR / "model_comparison_with_transaction_cost_model.csv", index=False)
comparison_table


## Final interpretation

This block generates a compact interpretation that can be adapted into the report.


In [ ]:
def build_tc_model_interpretation(comparison_table):
    rows = []
    rows.append(
        "The transaction-cost-aware model is trained by including implementation costs directly in the objective function. "
        "This differs from an ex-post cost analysis because the trading policy can adapt during training to reduce costly rebalancing."
    )

    if "transaction_cost_aware" in set(comparison_table["model"]):
        tc = comparison_table[comparison_table["model"] == "transaction_cost_aware"].iloc[0]
        rows.append(
            f"The transaction-cost-aware strategy reaches a net-error MSE of {tc['mse_net_error']:.4f}, "
            f"with average net error {tc['mean_net_error']:.4f} and 5% CVaR {tc['net_cvar05']:.4f}. "
            f"Its average transaction cost is {tc['mean_transaction_cost']:.4f}, with average turnover {tc['mean_turnover']:.4f}."
        )

    if "baseline_mse" in set(comparison_table["model"]) and "transaction_cost_aware" in set(comparison_table["model"]):
        base = comparison_table[comparison_table["model"] == "baseline_mse"].iloc[0]
        tc = comparison_table[comparison_table["model"] == "transaction_cost_aware"].iloc[0]
        rows.append(
            f"Compared with the MSE baseline, whose average transaction cost is {base['mean_transaction_cost']:.4f}, "
            f"the transaction-cost-aware model has average transaction cost {tc['mean_transaction_cost']:.4f}. "
            "The comparison indicates whether internalizing frictions during training effectively reduces trading intensity."
        )

    rows.append(
        "This experiment completes the analysis of market frictions by moving from ex-post measurement to a model whose objective explicitly accounts for transaction costs."
    )

    return "\n\n".join(rows)

tc_interpretation = build_tc_model_interpretation(comparison_table)
print(tc_interpretation)

with open(RESULTS_DIR / "transaction_cost_model_interpretation.txt", "w", encoding="utf-8") as f:
    f.write(tc_interpretation)


# Final outputs

This notebook saves:
- `deltas_transaction_cost.npy`,
- `net_errors_transaction_cost.npy`,
- `gross_errors_transaction_cost.npy`,
- `transaction_costs_model.npy`,
- `model_transaction_cost.pt`,
- comparison tables and interpretation files.

These outputs make it possible to complete the transaction-cost section of the project and to use the trained strategy in the financial evaluation notebook.
